# 🏥 Dashboard Interaktif Prediksi Biaya Asuransi Kesehatan
**Tugas Visualisasi Data** — Python + Streamlit + Plotly + Scikit-learn (dijalankan di Google Colab)

| Item | Keterangan |
|---|---|
| Dataset | Medical Cost Personal Dataset (1.338 baris, 7 kolom, tanpa missing value) |
| Link data | https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv |
| Sumber asli | https://github.com/stedy/Machine-Learning-with-R-datasets |
| Algoritma | Gradient Boosting Regressor (ringan, akurasi R² ≈ 0,88) |
| Layout | Pola **Z**, satu halaman |

**Alur notebook:** 1) Install → 2) Load & cek data → 3) Uji model cepat → 4) Tulis `app.py` → 5) Jalankan dashboard → 6) Deploy permanen ke Streamlit Cloud.

## 1. Install library

In [ ]:
!pip install -q -U streamlit plotly scikit-learn pandas

## 2. Load data & cek singkat (EDA)
Data sudah bersih, jadi tidak perlu proses cleaning yang rumit.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(URL)

print("Ukuran data :", df.shape)
print("Missing value:", df.isna().sum().sum())
display(df.head())
display(df.describe().round(2))

**Penjelasan kolom**

- `age` usia · `sex` jenis kelamin · `bmi` indeks massa tubuh · `children` jumlah anak
- `smoker` perokok (yes/no) · `region` wilayah di AS
- `charges` **target prediksi** — biaya medis tahunan (USD)

## 3. Uji model cepat (sebelum masuk dashboard)
Memastikan model cukup akurat. Hasilnya dipakai sebagai acuan di dashboard.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

X = pd.get_dummies(df.drop(columns="charges"))
y = df["charges"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

m = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.1, random_state=42)
m.fit(X_tr, y_tr)
pred = m.predict(X_te)
print(f"R²  : {r2_score(y_te, pred):.3f}")
print(f"MAE : ${mean_absolute_error(y_te, pred):,.0f}")

## 4. Tulis file `app.py` (dashboard versi profesional)
Berisi CSS kustom, kartu KPI, tema grafik Plotly, heatmap, insight otomatis, dan kartu hasil prediksi.

In [ ]:
%%writefile app.py
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------------
# 0. KONFIGURASI & TEMA
# ------------------------------------------------------------------
st.set_page_config(page_title="Insurance Cost Analytics", page_icon="🏥", layout="wide")

NAVY, TEAL, ORANGE, SLATE, GRID = "#0B2545", "#0FA3B1", "#F77F00", "#64748B", "#E8EDF4"
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
NUM, CAT = ["age", "bmi", "children"], ["sex", "smoker", "region"]
LABEL = {"age": "Usia", "bmi": "BMI", "children": "Jumlah Anak", "sex": "Jenis Kelamin",
         "smoker": "Perokok", "region": "Wilayah", "kelompok_bmi": "Kategori BMI"}
SMOKER_COLOR = {"yes": ORANGE, "no": TEAL}

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
html, body, [class*="css"], .stApp {font-family:'Inter',sans-serif;}
.stApp {background:#F4F7FB;}
#MainMenu, footer {visibility:hidden;}
header[data-testid="stHeader"] {background:#F4F7FB;}
:root {color-scheme: light;}
[data-testid="stToolbar"] *, [data-testid="stHeader"] * {color:#1E3A8A !important;}
.block-container {padding-top:3rem; padding-bottom:2rem; max-width:1400px;}
h1,h2,h3,h4 {color:#0B2545; letter-spacing:-0.02em;}

/* Hero */
.hero {background:linear-gradient(135deg,#0B2545 0%,#13315C 55%,#0FA3B1 130%);
  border-radius:18px; padding:28px 32px; color:#fff; height:100%;
  box-shadow:0 8px 24px rgba(11,37,69,.22);}
.hero h1 {color:#fff; font-size:1.9rem; font-weight:800; margin:6px 0 8px 0; line-height:1.15;}
.hero p {color:#CFE3F1; font-size:.92rem; margin:0 0 14px 0; line-height:1.5;}
.badge {display:inline-block; background:rgba(255,255,255,.14); border:1px solid rgba(255,255,255,.25);
  color:#fff; padding:3px 11px; border-radius:999px; font-size:.72rem; font-weight:600; margin-right:6px;}
.eyebrow {font-size:.72rem; font-weight:700; letter-spacing:.14em; color:#7FD8E0;}

/* Section label */
.sec {display:flex; align-items:center; gap:10px; margin:18px 0 8px 0;}
.sec .num {background:#0B2545; color:#fff; width:26px; height:26px; border-radius:50%;
  display:flex; align-items:center; justify-content:center; font-size:.8rem; font-weight:700;}
.sec .ttl {font-size:1.05rem; font-weight:700; color:#0B2545;}
.sec .sub {font-size:.82rem; color:#64748B; margin-left:4px;}

/* Card container */
div[class*="st-key-card"] {background:#fff; border:1px solid #E5EAF2 !important;
  border-radius:16px !important; padding:6px 10px; box-shadow:0 2px 12px rgba(15,37,69,.05);}

/* KPI */
.kpi {background:#fff; border-radius:14px; padding:16px 18px; border:1px solid #E5EAF2;
  border-left:5px solid var(--accent); box-shadow:0 2px 12px rgba(15,37,69,.05);}
.kpi-label {font-size:.72rem; font-weight:700; letter-spacing:.08em; color:#64748B; text-transform:uppercase;}
.kpi-value {font-size:clamp(1.3rem,2vw,1.75rem); white-space:nowrap; font-weight:800; color:#0B2545; margin:2px 0; line-height:1.15;}
.kpi-sub {font-size:.78rem; color:#64748B;}

/* Insight */
.insight {background:#ECFEFF; border:1px solid #BDEFF4; border-radius:12px; padding:12px 16px;
  color:#0B2545; font-size:.9rem; margin:12px 0 4px 0;}

/* Result */
.result {background:linear-gradient(135deg,#0B2545,#1D4ED8 120%); border-radius:16px; padding:20px 26px;
  color:#fff; box-shadow:0 8px 24px rgba(29,78,216,.28); display:flex; justify-content:space-between;
  align-items:center; flex-wrap:wrap; gap:14px; margin:14px 0 6px 0;}
.r-label {font-size:.72rem; font-weight:700; letter-spacing:.14em; color:#BFD6FF;}
.r-value {font-size:clamp(2rem,3.4vw,2.8rem); font-weight:800; line-height:1.1; margin:4px 0 2px 0; white-space:nowrap; color:#fff;}
.r-idr {font-size:1rem; color:#E0ECFF;}
.r-badge {display:inline-block; background:rgba(255,255,255,.18); padding:6px 14px;
  border-radius:999px; font-size:.85rem; font-weight:600; color:#fff;}
.r-note {color:#DCE9FF; font-size:.82rem; margin-top:8px;}

/* ===== Perbaikan keterbacaan (aman untuk mode gelap & terang) ===== */
.stApp [data-testid="stMarkdownContainer"] p,
.stApp [data-testid="stMarkdownContainer"] li,
.stApp [data-testid="stMarkdownContainer"] strong,
.stApp [data-testid="stMarkdownContainer"] em {color:#1E3A8A;}
.stApp .hero p {color:#CFE3F1 !important;}
.stApp .hero h1 {color:#fff !important;}
.stApp [data-testid="stWidgetLabel"], .stApp [data-testid="stWidgetLabel"] * {color:#1E3A8A !important; font-weight:600;}
.stApp [data-testid="stCaptionContainer"], .stApp [data-testid="stCaptionContainer"] * {color:#475569 !important;}
.stApp [data-testid="stRadio"] label, .stApp [data-testid="stRadio"] label * {color:#1E3A8A !important;}
.stApp [data-testid="stExpander"] summary, .stApp [data-testid="stExpander"] summary * {color:#1E3A8A !important; font-weight:600;}
.stApp [data-testid="stExpander"] details {background:#fff; border:1px solid #E5EAF2; border-radius:12px;}
/* Tab */
.stApp [data-baseweb="tab-list"] button p {color:#475569 !important; font-weight:600;}
.stApp [data-baseweb="tab-list"] button[aria-selected="true"] p {color:#1D4ED8 !important;}
.stApp [data-baseweb="tab-highlight"] {background:#1D4ED8 !important;}
/* Dropdown / multiselect / input jadi terang */
.stApp div[data-baseweb="select"] > div, .stApp div[data-baseweb="input"] > div, .stApp div[data-baseweb="base-input"] {
  background:#fff !important; border:1px solid #CBD5E1 !important; border-radius:10px !important;}
.stApp div[data-baseweb="select"] *, .stApp input {color:#0B2545 !important;}
.stApp div[data-baseweb="select"] svg {fill:#1E3A8A !important;}
.stApp span[data-baseweb="tag"] {background:#1D4ED8 !important;}
.stApp span[data-baseweb="tag"] * {color:#fff !important;}
div[data-baseweb="popover"] ul, div[data-baseweb="popover"] li {background:#fff !important; color:#0B2545 !important;}
div[data-baseweb="popover"] li:hover {background:#E6F0FF !important;}
/* Slider */
.stApp [data-testid="stSlider"] [data-testid="stThumbValue"], .stApp [data-testid="stSlider"] [data-testid="stTickBarMin"],
.stApp [data-testid="stSlider"] [data-testid="stTickBarMax"] {color:#1D4ED8 !important; font-weight:700;}
.stApp [data-testid="stSlider"] div[role="slider"] {background:#1D4ED8 !important; border-color:#1D4ED8 !important;}

/* Brute-force: semua kotak input terang */
.stApp [data-baseweb="select"], .stApp [data-baseweb="select"] > div, .stApp [data-baseweb="select"] > div > div,
.stApp [data-baseweb="input"], .stApp [data-baseweb="input"] > div, .stApp [data-baseweb="base-input"],
.stApp [data-testid="stNumberInput"] input {background:#fff !important; color:#0B2545 !important; -webkit-text-fill-color:#0B2545 !important;}
.stApp [data-baseweb="select"] > div {border:1px solid #CBD5E1 !important; border-radius:10px !important;}
.stApp [data-baseweb="select"] span, .stApp [data-baseweb="select"] div[value] {color:#0B2545 !important;}
.stApp [data-baseweb="select"] span[data-baseweb="tag"], .stApp [data-baseweb="select"] span[data-baseweb="tag"] > span {background:#1D4ED8 !important; color:#fff !important;}
.stApp [data-testid="stNumberInputStepDown"], .stApp [data-testid="stNumberInputStepUp"] {background:#EEF2FF !important; color:#1E3A8A !important;}
.stApp [data-testid="stNumberInputStepDown"] *, .stApp [data-testid="stNumberInputStepUp"] * {color:#1E3A8A !important; fill:#1E3A8A !important;}
div[data-baseweb="popover"], div[data-baseweb="popover"] > div, div[data-baseweb="menu"], div[data-baseweb="menu"] * {background:#fff !important; color:#0B2545 !important;}
div[data-baseweb="menu"] li:hover, div[data-baseweb="menu"] li[aria-selected="true"] {background:#E6F0FF !important;}
/* Radio & slider fallback */
.stApp [data-testid="stRadio"] [role="radio"][aria-checked="true"] > div:first-child {background:#1D4ED8 !important; border-color:#1D4ED8 !important;}
.stApp [data-testid="stRadio"] [role="radio"][aria-checked="false"] > div:first-child {background:#fff !important; border:2px solid #94A3B8 !important;}
.stApp [data-testid="stSlider"] [data-baseweb="slider"] > div > div {background:linear-gradient(to right,#1D4ED8,#1D4ED8) !important;}

/* Select: paksa semua level terang */
.stApp [data-testid="stSelectbox"] [data-baseweb="select"] div,
.stApp [data-testid="stMultiSelect"] [data-baseweb="select"] div {background-color:#fff !important; background-image:none !important; color:#0B2545 !important;}
.stApp [data-testid="stMultiSelect"] span[data-baseweb="tag"] {background-color:#1D4ED8 !important;}
.stApp [data-testid="stMultiSelect"] span[data-baseweb="tag"] * {background-color:transparent !important; color:#fff !important;}
.stApp [data-testid="stSelectbox"] [data-baseweb="select"] > div {border:1px solid #CBD5E1 !important; border-radius:10px !important; min-height:42px;}

/* Tabs & widgets */
button[data-baseweb="tab"] {font-weight:600;}
.foot {text-align:center; color:#94A3B8; font-size:.78rem; margin-top:24px;}
</style>
""", unsafe_allow_html=True)


def kpi(label, value, sub, color):
    return (f'<div class="kpi" style="--accent:{color}"><div class="kpi-label">{label}</div>'
            f'<div class="kpi-value">{value}</div><div class="kpi-sub">{sub}</div></div>')


def section(num, title, sub=""):
    st.markdown(f'<div class="sec"><div class="num">{num}</div><div class="ttl">{title}</div>'
                f'<div class="sub">{sub}</div></div>', unsafe_allow_html=True)


def style(fig, h=360, legend=True):
    has_title = bool(fig.layout.title.text)
    top = 56 if (legend or has_title) else 24
    fig.update_layout(
        height=h, margin=dict(t=top, b=48, l=64, r=24),
        font=dict(family="Inter, sans-serif", size=12, color=NAVY),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        hoverlabel=dict(bgcolor="white", font_size=12, bordercolor=GRID),
        showlegend=legend,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, title_text=""),
        title=dict(x=0, xanchor="left", y=0.97, font=dict(size=14, color=NAVY)))
    fig.update_xaxes(showgrid=False, linecolor=GRID, zeroline=False, automargin=True,
                     title_standoff=14, tickfont=dict(color=SLATE), title_font=dict(size=12, color=NAVY))
    fig.update_yaxes(gridcolor=GRID, zeroline=False, automargin=True,
                     title_standoff=14, tickfont=dict(color=SLATE), title_font=dict(size=12, color=NAVY))
    return fig


def rupiah(x):
    return "Rp " + f"{x:,.0f}".replace(",", ".")

# ------------------------------------------------------------------
# 1. DATA & MODEL
# ------------------------------------------------------------------
@st.cache_data
def load_data():
    df = pd.read_csv(URL)
    df["kelompok_bmi"] = pd.cut(df["bmi"], [0, 18.5, 25, 30, 100],
                                labels=["Kurus", "Normal", "Gemuk", "Obesitas"])
    df["kelompok_usia"] = pd.cut(df["age"], [17, 25, 35, 45, 55, 64],
                                 labels=["18-25", "26-35", "36-45", "46-55", "56-64"])
    return df


@st.cache_resource
def train_model(n_estimators, max_depth, learning_rate):
    df = load_data()
    X, y = df[NUM + CAT], df["charges"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)],
                            remainder="passthrough")
    model = Pipeline([("pre", pre),
                      ("gb", GradientBoostingRegressor(n_estimators=n_estimators, max_depth=max_depth,
                                                       learning_rate=learning_rate, random_state=42))])
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    names = model.named_steps["pre"].get_feature_names_out()
    imp = model.named_steps["gb"].feature_importances_
    agg = {}
    for n, v in zip(names, imp):
        col = n.split("__")[1]
        for c in CAT:
            if col.startswith(c + "_"):
                col = c
        agg[LABEL[col]] = agg.get(LABEL[col], 0) + v
    imp_df = pd.DataFrame({"Fitur": list(agg), "Pengaruh": list(agg.values())}).sort_values("Pengaruh")
    return model, r2_score(y_te, pred), mean_absolute_error(y_te, pred), imp_df


df = load_data()

# ==================================================================
# LAYOUT POLA Z
#  Kiri-atas (hero) ──► Kanan-atas (filter)
#        ╲  diagonal: KPI + grafik eksplorasi
#  Kiri-bawah (model) ──► Kanan-bawah (PREDIKSI)
# ==================================================================

# ---------- BARIS 1 ----------
c_hero, c_filter = st.columns([1.25, 1.75], gap="large")
with c_hero:
    st.markdown("""
    <div class="hero">
      <div class="eyebrow">DATA ANALYTICS DASHBOARD</div>
      <h1>Insurance Cost Analytics &amp; Prediction</h1>
      <p>Eksplorasi faktor yang memengaruhi biaya asuransi kesehatan dan simulasikan
      prediksinya secara interaktif. Mulai dari filter ➜ baca insight ➜ coba prediksi.</p>
      <span class="badge">1.338 peserta</span><span class="badge">Gradient Boosting</span>
      <span class="badge">Plotly + Streamlit</span>
    </div>""", unsafe_allow_html=True)
with c_filter:
    with st.container(border=True, key="card_filter"):
        st.markdown("**🎛️ Filter Data** · berlaku untuk KPI & grafik di bawah")
        f1, f2, f3 = st.columns(3)
        sm = f1.multiselect("Status perokok", ["yes", "no"], default=["yes", "no"])
        rg = f2.multiselect("Wilayah", sorted(df.region.unique()), default=sorted(df.region.unique()))
        ag = f3.slider("Rentang usia", 18, 64, (18, 64))
        st.caption("Tips: pilih hanya *yes* pada perokok untuk melihat segmen biaya tertinggi.")

d = df[df.smoker.isin(sm) & df.region.isin(rg) & df.age.between(*ag)]
if d.empty:
    st.warning("Tidak ada data untuk kombinasi filter ini. Silakan ubah filter.")
    st.stop()

# ---------- DIAGONAL : KPI ----------
section(1, "Ringkasan", "gambaran umum data terfilter")
k1, k2, k3, k4 = st.columns(4)
k1.markdown(kpi("Jumlah Peserta", f"{len(d):,}", f"{len(d)/len(df)*100:.0f}% dari total data", NAVY), unsafe_allow_html=True)
k2.markdown(kpi("Rata-rata Biaya", f"${d.charges.mean():,.0f}", f"Median ${d.charges.median():,.0f}", TEAL), unsafe_allow_html=True)
k3.markdown(kpi("Biaya Tertinggi", f"${d.charges.max():,.0f}", f"Terendah ${d.charges.min():,.0f}", ORANGE), unsafe_allow_html=True)
k4.markdown(kpi("Proporsi Perokok", f"{(d.smoker == 'yes').mean()*100:.0f}%", f"Rata-rata BMI {d.bmi.mean():.1f}", SLATE), unsafe_allow_html=True)

# Insight otomatis
if d.smoker.nunique() == 2:
    ratio = d[d.smoker == "yes"].charges.mean() / d[d.smoker == "no"].charges.mean()
    st.markdown(f'<div class="insight">💡 <b>Insight:</b> pada data terfilter, peserta perokok membayar rata-rata '
                f'<b>{ratio:.1f}× lebih tinggi</b> dibanding non-perokok.</div>', unsafe_allow_html=True)

# ---------- DIAGONAL : GRAFIK ----------
section(2, "Eksplorasi", "temukan pola dengan mengubah pilihan di tiap grafik")
g1, g2 = st.columns(2, gap="large")
with g1:
    with st.container(border=True, key="card_g1"):
        st.markdown("**Hubungan variabel dengan biaya**")
        x_var = st.radio("Sumbu X", ["age", "bmi"], format_func=lambda v: LABEL[v], horizontal=True,
                         label_visibility="collapsed")
        fig = px.scatter(d, x=x_var, y="charges", color="smoker", opacity=0.75,
                         hover_data=["sex", "region", "children"], color_discrete_map=SMOKER_COLOR,
                         labels={x_var: LABEL[x_var], "charges": "Biaya (USD)", "smoker": "Perokok"})
        fig.update_traces(marker=dict(size=7, line=dict(width=0.5, color="white")))
        st.plotly_chart(style(fig, 400), width="stretch", theme=None)
with g2:
    with st.container(border=True, key="card_g2"):
        t1, t2 = st.tabs(["Rata-rata per kelompok", "Heatmap usia × BMI"])
        with t1:
            grp = st.selectbox("Kelompokkan berdasarkan", ["region", "sex", "smoker", "children", "kelompok_bmi"],
                               format_func=lambda v: LABEL[v])
            a = d.groupby(grp, observed=True)["charges"].mean().reset_index()
            fig2 = px.bar(a, x=grp, y="charges", text_auto=".2s", labels={grp: LABEL[grp], "charges": "Rata-rata biaya (USD)"})
            fig2.update_traces(marker_color=TEAL, marker_line_width=0, textposition="outside", cliponaxis=False)
            st.plotly_chart(style(fig2, 340, legend=False), width="stretch", theme=None)
        with t2:
            hm = d.pivot_table(index="kelompok_usia", columns="kelompok_bmi", values="charges",
                               aggfunc="mean", observed=False)
            fig4 = px.imshow(hm, text_auto=".0f", aspect="auto", color_continuous_scale=["#E6F7F9", TEAL, NAVY],
                             labels=dict(x="Kategori BMI", y="Kelompok usia", color="USD"))
            fig4.update_layout(coloraxis_showscale=False)
            st.plotly_chart(style(fig4, 400, legend=False), width="stretch", theme=None)

# ---------- BARIS 3 : MODEL (kiri) ──► PREDIKSI (kanan) ----------
section(3, "Model & Prediksi", "atur parameter lalu lihat hasilnya langsung berubah")
left, right = st.columns([1, 1.7], gap="large")

with left:
    with st.container(border=True, key="card_model"):
        st.markdown("**🤖 Performa model**")
        with st.expander("⚙️ Atur parameter model"):
            n_est = st.slider("Jumlah pohon (n_estimators)", 50, 400, 150, 50)
            depth = st.slider("Kedalaman pohon (max_depth)", 1, 6, 3)
            lr = st.select_slider("Learning rate", [0.01, 0.05, 0.1, 0.2, 0.3], value=0.1)
        model, r2, mae, imp_df = train_model(n_est, depth, lr)
        m1, m2 = st.columns(2)
        m1.markdown(kpi("Akurasi R²", f"{r2:.2f}", "makin dekat 1 makin baik", TEAL), unsafe_allow_html=True)
        m2.markdown(kpi("Rata-rata galat", f"${mae:,.0f}", "selisih rata-rata (MAE)", ORANGE), unsafe_allow_html=True)
        fi = px.bar(imp_df, x="Pengaruh", y="Fitur", orientation="h", title="Faktor paling berpengaruh pada biaya")
        fi.update_traces(marker_color=[TEAL] * (len(imp_df) - 1) + [ORANGE], marker_line_width=0)
        fi.update_yaxes(showgrid=False)
        fi.update_xaxes(showgrid=True, gridcolor=GRID)
        fi.update_xaxes(tickformat=".0%")
        st.plotly_chart(style(fi, 330, legend=False), width="stretch", theme=None)
        top_f = imp_df.iloc[-1]
        st.markdown(f'<div class="insight">💡 <b>{top_f["Fitur"]}</b> menjelaskan sekitar '
                    f'<b>{top_f["Pengaruh"]*100:.0f}%</b> perbedaan biaya antar peserta menurut model.</div>',
                    unsafe_allow_html=True)
        st.caption(f"Parameter aktif: n_estimators={n_est} · max_depth={depth} · learning_rate={lr}")

with right:
    with st.container(border=True, key="card_pred"):
        st.markdown("**🔮 Simulasi prediksi biaya** · geser parameter, hasil langsung berubah")
        a1, a2, a3 = st.columns(3)
        age = a1.slider("Usia", 18, 64, 35)
        bmi = a2.slider("BMI", 15.0, 50.0, 27.0, 0.5)
        kids = a3.slider("Jumlah anak", 0, 5, 1)
        b1, b2, b3, b4 = st.columns([1, 1.2, 1.2, 1])
        smoker = b1.radio("Perokok?", ["no", "yes"], horizontal=True)
        sex = b2.radio("Jenis kelamin", ["female", "male"], horizontal=True)
        region = b3.selectbox("Wilayah", sorted(df.region.unique()))
        kurs = b4.number_input("Kurs USD → IDR", 10000, 20000, 16000, 100)

        row = pd.DataFrame([{"age": age, "bmi": bmi, "children": kids, "sex": sex, "smoker": smoker, "region": region}])
        hasil = model.predict(row)[0]
        pct = (df.charges < hasil).mean() * 100
        mean_all = df.charges.mean()
        sel = (hasil - mean_all) / mean_all * 100
        st.markdown(f'<div class="result"><div><div class="r-label">PREDIKSI BIAYA PER TAHUN</div>'
                    f'<div class="r-value">${hasil:,.0f}</div><div class="r-idr">≈ {rupiah(hasil * kurs)}</div></div>'
                    f'<div><div class="r-badge">Lebih mahal dari {pct:.0f}% peserta</div>'
                    f'<div class="r-note">{abs(sel):.0f}% {"di atas" if sel > 0 else "di bawah"} rata-rata semua peserta (${mean_all:,.0f})</div></div>'
                    f'</div>', unsafe_allow_html=True)

        p1, p2 = st.tabs(["Posisi di distribusi biaya", "Simulasi biaya vs usia"])
        with p1:
            fig5 = px.histogram(df, x="charges", nbins=40, labels={"charges": "Biaya (USD)"})
            fig5.update_traces(marker_color="#BFD6FF", marker_line_width=0)
            pos_pred = "top right" if hasil >= mean_all else "top left"
            pos_mean = "top left" if hasil >= mean_all else "top right"
            fig5.add_vline(x=hasil, line_color=ORANGE, line_width=3, annotation_text="Prediksi Anda",
                           annotation_position=pos_pred, annotation_font_color=ORANGE)
            fig5.add_vline(x=mean_all, line_dash="dash", line_color=NAVY, annotation_text="Rata-rata",
                           annotation_position=pos_mean, annotation_font_color=NAVY)
            fig5.update_yaxes(title="Jumlah peserta")
            st.plotly_chart(style(fig5, 280, legend=False), width="stretch", theme=None)
        with p2:
            ages = list(range(18, 65, 2))
            fig3 = go.Figure()
            for s_, col, nm in [("no", TEAL, "Non-perokok"), ("yes", ORANGE, "Perokok")]:
                grid = pd.DataFrame({"age": ages, "bmi": bmi, "children": kids, "sex": sex, "smoker": s_, "region": region})
                fig3.add_scatter(x=ages, y=model.predict(grid), mode="lines", name=nm, line=dict(color=col, width=3))
            fig3.add_scatter(x=[age], y=[hasil], mode="markers", name="Input Anda",
                             marker=dict(size=15, color=NAVY, symbol="star", line=dict(width=2, color="white")))
            fig3.update_layout(xaxis_title="Usia", yaxis_title="Prediksi biaya (USD)")
            st.plotly_chart(style(fig3, 280), width="stretch", theme=None)

st.markdown('<div class="foot">Sumber data: Medical Cost Personal Dataset (Brett Lantz, <i>Machine Learning with R</i>) · '
            'github.com/stedy/Machine-Learning-with-R-datasets</div>', unsafe_allow_html=True)

Buat juga `requirements.txt` (dipakai saat deploy ke Streamlit Cloud):

In [ ]:
%%writefile requirements.txt
streamlit>=1.50
plotly
scikit-learn
pandas

Tema warna dashboard (opsional tapi disarankan, agar tampilan konsisten di semua perangkat):

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#1D4ED8"
backgroundColor = "#F4F7FB"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#0B2545"
font = "sans serif"

[browser]
gatherUsageStats = false

## 5. Jalankan dashboard dari Colab
Colab tidak punya alamat publik, jadi kita pakai **localtunnel**.

1. Jalankan sel di bawah.
2. Salin **IP** yang muncul (itu password-nya).
3. Klik link `https://xxxx.loca.lt`, lalu tempel IP tadi di kolom *Tunnel Password*.

In [ ]:
!npm install -s localtunnel
!streamlit run app.py --server.port 8501 &>/content/logs.txt &
import time; time.sleep(5)
print("Password tunnel (salin IP ini):")
!curl -s https://ipv4.icanhazip.com
!npx localtunnel --port 8501

> Jika localtunnel bermasalah, gunakan alternatif Cloudflare:
> ```python
> !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
> !./cloudflared tunnel --url http://localhost:8501
> ```
> Cari link berakhiran `.trycloudflare.com` di output.

## 6. Deployment permanen ke Streamlit Community Cloud (gratis)
Link Colab/localtunnel hanya aktif selama notebook berjalan. Agar dashboard online terus dan bisa dikumpulkan sebagai tugas:

1. **Download** `app.py`, `requirements.txt`, dan `.streamlit/config.toml` dari panel file Colab (ikon 📁 di kiri → klik kanan → Download).
2. Buat akun di https://github.com → **New repository** (misal `dashboard-asuransi`, Public).
3. Klik **Add file → Upload files**, unggah `app.py` dan `requirements.txt`, lalu **Commit**.
4. Buka https://share.streamlit.io → login dengan GitHub → **Create app**.
5. Pilih repository tadi, *Main file path* isi `app.py` → **Deploy**.
6. Tunggu 2–5 menit. Anda mendapat link publik `https://namaapp.streamlit.app`.

## 7. Catatan untuk laporan / presentasi
- **Pola Z:** judul (kiri-atas) → filter (kanan-atas) → KPI & grafik (diagonal) → akurasi model (kiri-bawah) → prediksi (kanan-bawah).
- **Interaktif:** filter data, pilihan sumbu/kelompok grafik, slider prediksi, dan parameter model (`n_estimators`, `max_depth`, `learning_rate`).
- **Insight utama:** status perokok adalah faktor paling berpengaruh pada biaya, diikuti BMI dan usia.
- **Keterbatasan:** data berasal dari AS (USD), 1.338 sampel, hanya 6 fitur.